In [ ]:
!pip install selenium webdriver-manager

# Web Scrapping

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

def format_number(n):
    """Formatea un número con puntos como separador de miles"""
    try:
        n_int = int(n)
        return f"{n_int:,}".replace(",", ".")
    except:
        return n

def get_todos_jugadores():
    # Configuració del driver
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    driver.get("https://www.futbolfantasy.com/analytics/laliga-fantasy/mercado") # aqui pot ser hauria de posar una altre URL  per poder extreure màxim bid possible?

    time.sleep(3)  # Esperar que carregui la pàgina i el contingut dinàmic 

    elementos = driver.find_elements(By.CSS_SELECTOR, "div.elemento_jugador")
    resultados = []

    for elem in elementos:
        nombre = elem.get_attribute("data-nombre").title()
        info = {
            "jugador": nombre,
            "diferencia_valor": elem.get_attribute("data-diferencia1"),
            "valor_actual": elem.get_attribute("data-valor"),
            "valor_anterior": elem.get_attribute("data-valor1"),
            "puja_maxima_rentable": elem.get_attribute("data-valor7")
        }
        resultados.append(info)

    driver.quit()

    # creo df
    df = pd.DataFrame(resultados, columns=[
        "jugador",
        "diferencia_valor",
        "valor_actual",
        "valor_anterior",
        "puja_maxima_rentable"
    ])

    # Formatting
    for col in ["diferencia_valor", "valor_actual", "valor_anterior", "puja_maxima_rentable"]:
        df[col] = df[col].apply(format_number)

    return df


if __name__ == "__main__":
    df = get_todos_jugadores()


df


,jugador,diferencia_valor,valor_actual,valor_anterior,puja_maxima_rentable
0,Antony,3.538.572,72.380.113,68.841.541,0
1,Joan Garcia,1.932.642,61.848.369,59.915.727,52.916.313
2,Nicolas Pepe,1.889.915,35.176.521,33.286.606,23.416.133
3,Vinicius Junior,1.758.978,111.716.407,109.957.429,102.494.422
4,Arda Guler,1.699.851,87.166.587,85.466.736,79.752.866
...,...,...,...,...,...
595,Isco Alarcon,-1.226.831,40.741.893,41.968.724,50.731.491
596,Alejandro Balde,-1.256.350,52.679.857,53.936.207,60.355.527
597,Pau Cubarsi,-1.493.178,63.677.861,65.171.039,72.442.549
598,Alex Baena,-1.566.287,53.946.766,55.513.053,65.711.090


# Define the players

In [ ]:
# Diccionari dels jugadors (ara mateix top4). Els noms s'han de posar exactament com surt a la pàgina web ja que sinó, no te'l troba. Vaig probar de fer un %like% però em surtien coses rares per jugadors tipo Junior Firpo i Vinicius Junior i Luiz Junior.
propietarios = {
    "Adri": [
        "Joan García",
        "Matteo Ruggeri",
        "Juan Foyth",
        "Aitor Paredes",
        "Yoel Lago",
        "Antonio Raíllo",
        "Santi Comesaña",
        "Ramon Terrats",
        "Thiago Almada",
        "Rafa Mir",
        "Roberto Fernandez",
        "Isi Palazon",
        "Yaser Asprilla"
    ],
    "Carre": [
        "Antonio Sivera",
        "Alvaro F. Carreras",
        "Yuri Berchiche",
        "Dean Huijsen",
        "Facundo Garcés",
        "Rafa Marín",
        "Victor Parada",
        "Luiz Junior",
        "Dakonam Djene",
        "Luis Milla",
        "Hugo Alvarez",
        "Fermin López",
        "Etta Eyong",
        "Carlos Vicente",
        "Giuliano Simeone",
        "Franco Mastantuono",
        "Álvaro García"
    ],
    "Enric": [
        "Jose Angel Carmona",
        "Sergi Cardona",
        "Santiago Mouriño",
        "Cesar Tárrega",
        "Jose Manuel Copete",
        "Valentin Rosier",
        "Jorge Cabello",
        "Edu Expósito",
        "Javi Puado",
        "Alfonso Gonzalez",
        "Raphinha",
        "Marcos Alonso",
        "Marko Dmitrovic"

    ],
    "Swedish": [
        "Aarón Escandell",
        "Leo Román",
        "Álvaro Nuñez",
        "Kike Salas",
        "David Affengruber",
        "Jorge de Frutos",
        "Carles Aleñá",
        "Adrián Liso",
        "Sergi Altimira",
        "Kylian Mbappé",
        "Orri Steinn Oskarsson",
        "Nicolas Pepe",
        "Abdel Abqar"
    ]
}


# Save csv automatically - historical data

In [ ]:
import os
from datetime import datetime
import unidecode  # pip install unidecode si no ho tens

# Strip de noms per treure accents i majúsucules
def normalize(name):
    return unidecode.unidecode(name.strip().lower())


# creo columna de propietari per poder omplir
df['propietario'] = ""

# Assignació de jugadors
for propietario, lista in propietarios.items():
    lista_normalizada = [normalize(j) for j in lista]
    for jugador_prop in lista_normalizada:
        mask = df['jugador'].apply(normalize).str.fullmatch(jugador_prop)
        df.loc[mask, 'propietario'] = propietario

# Afegixo data d'avui (quan s'executa)
from datetime import datetime
hoy = datetime.today().strftime("%Y-%m-%d")
df['fecha'] = hoy

# Path d'on guardar el csv
carpeta_data = os.path.join(os.getcwd(), "Data")
os.makedirs(carpeta_data, exist_ok=True)

# Nom del csv amb el que vull que es guardi
nombre_csv = f"data_{hoy}.csv"
ruta_csv = os.path.join(carpeta_data, nombre_csv)

# Fer overwrite en cas que s'executi varis cops el codi el mateix dia. I.e: si executo el codi avui 2 vegades, únicament tindré un fitxer guardat amb la última execució com a output.
df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

print(f"CSV guardado en: {ruta_csv}")


CSV guardado en: c:\Users\Adria Armengol\Analysis\Personal stuff\Data\data_2025-09-04.csv


# Format purposes

In [ ]:
import pandas as pd

# Columnas numéricas originales
cols_valores = ['diferencia_valor', 'valor_actual', 'valor_anterior', 'puja_maxima_rentable']

# Convertir a números
for col in cols_valores:
    df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Crear columnas formateadas solo para mostrar
for col in cols_valores:
    df[col + "_fmt"] = df[col].apply(lambda x: f"{x:,}".replace(",", "."))

# Ejemplo de uso
print(df[['jugador', 'valor_actual', 'valor_actual_fmt']].head(10))

# Ordenamiento correcto por valor_actual (numérico)
df_sorted = df.sort_values(by='valor_actual', ascending=False)


              jugador  valor_actual valor_actual_fmt
0              Antony      72380113       72.380.113
1         Joan Garcia      61848369       61.848.369
2        Nicolas Pepe      35176521       35.176.521
3     Vinicius Junior     111716407      111.716.407
4          Arda Guler      87166587       87.166.587
5  Franco Mastantuono      29041947       29.041.947
6         Isi Palazon      33705376       33.705.376
7        Dean Huijsen      99011197       99.011.197
8          Etta Eyong      19496785       19.496.785
9        Lamine Yamal     145535020      145.535.020


# Team evaluation

In [ ]:
# Agafar jugadors amb equop únicament
df_validos = df[(df['propietario'].notna()) & (df['propietario'] != "")]

# crear df agregat
df_agg = df_validos.groupby('propietario')[['jugador','valor_actual','diferencia_valor']].agg({
    'jugador': 'count',
    'valor_actual': 'sum',
    'diferencia_valor': 'sum'
}).reset_index()

# Format purposes
cols_valores = ['valor_actual', 'diferencia_valor']
for col in cols_valores:
    df_agg[col] = df_agg[col].apply(lambda x: f"{x:,}".replace(",", "."))

df_agg


,propietario,jugador,valor_actual,diferencia_valor
0,Adri,13,218.974.407,4.777.247
1,Carre,17,387.863.130,9.808.640
2,Enric,13,267.994.112,1.643.601
3,Swedish,13,266.336.281,5.452.807


# risk ratio & efficiency



In [ ]:
import pandas as pd
import plotly.express as px

# calcul de mètriques per valorar risk i eficiencia
df_stats = (
    df_validos.groupby("propietario")
    .apply(lambda g: pd.Series({
        "media_dif": g["diferencia_valor"].mean(),
        "std_dif": g["diferencia_valor"].std(),
        "valor_medio": g["valor_actual"].mean(),
        "valor_total": g["valor_actual"].sum()
    }))
    .reset_index()
)

# risk = std / |media|
df_stats["riesgo"] = df_stats["std_dif"] / df_stats["media_dif"].abs()

# efficiency = media diferencia / media valor
df_stats["eficiencia"] = df_stats["media_dif"] / df_stats["valor_medio"]

# --- gráfico ---
fig = px.scatter(
    df_stats,
    x="riesgo",
    y="eficiencia",
    size="valor_total",            # tamaño = valor total plantilla
    color="propietario",           # color = propietario
    hover_data=["media_dif", "std_dif", "valor_medio", "valor_total"],
    title="Efficiency & risk map"
)

fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig.show()


# best players by team

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_validos,
    x="valor_actual",
    y="diferencia_valor",
    color="propietario",
    hover_data=["jugador", "valor_actual", "diferencia_valor_fmt"],
    title="Rentabilitat vs Valor actual"
)

fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()


# Best players in the market (WIP)

In [ ]:
# Filtrar propietarios free players
df_free = df.query('propietario == ""')

fig = px.scatter(
    df_free,
    x="valor_actual",
    y="diferencia_valor",
    color="propietario",
    hover_data=["jugador", "valor_actual", "diferencia_valor_fmt"],
    title="Rentabilitat vs Valor actual"
)

fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()

In [ ]:
import numpy as np
import plotly.express as px

# --- métricas básicas ---
df_free["eficiencia"] = df_free["diferencia_valor"] / df_free["valor_actual"]
df_free["score_interes"] = df_free["eficiencia"] * np.log1p(df_free["valor_actual"])

# --- separar positivos y negativos ---
df_free_pos = df_free[df_free["diferencia_valor"] > 0].copy()
df_free_neg = df_free[df_free["diferencia_valor"] < 0].copy()

# --- top 10 mejores ---
df_top_pos = (
    df_free_pos.sort_values("score_interes", ascending=False)
    .head(10)
)

# --- top 10 peores ---
df_top_neg = (
    df_free_neg.sort_values("score_interes", ascending=True)  # más negativos arriba
    .head(10)
)

print("🔝 Mejores jugadores libres para fichar:")
print(df_top_pos[["jugador", "valor_actual", "diferencia_valor", "eficiencia", "score_interes"]])

print("\n🔻 Peores jugadores libres (bajando):")
print(df_top_neg[["jugador", "valor_actual", "diferencia_valor", "eficiencia", "score_interes"]])

# --- gráfico barras positivos ---
fig_pos = px.bar(
    df_top_pos,
    x="score_interes",
    y="jugador",
    orientation="h",
    hover_data=["valor_actual", "diferencia_valor", "eficiencia"],
    title="Top 10 jugadores libres con mayor interés (positivos)",
)
fig_pos.update_layout(yaxis=dict(categoryorder="total ascending"))
fig_pos.show()

# --- gráfico barras negativos ---
fig_neg = px.bar(
    df_top_neg,
    x="score_interes",
    y="jugador",
    orientation="h",
    hover_data=["valor_actual", "diferencia_valor", "eficiencia"],
    title="Top 10 jugadores libres en caída (negativos)",
)
fig_neg.update_layout(yaxis=dict(categoryorder="total ascending"))
fig_neg.show()


🔝 Mejores jugadores libres para fichar:
              jugador  valor_actual  diferencia_valor  eficiencia  \
62         Javi Rueda       2435502            249867    0.102594   
64        Hugo Sotelo       2870914            245416    0.085484   
10     Tajon Buchanan      14338053           1054207    0.073525   
44      German Valera       5074644            389787    0.076811   
54   Alvaro Rodriguez       4739306            324084    0.068382   
47         Pere Milla       5926714            383900    0.064775   
77    Rodrigo Mendoza       2412470            165367    0.068547   
61        Pedro Bigas       4318250            258997    0.059977   
32  Ander Barrenetxea       9039982            501154    0.055437   
0              Antony      72380113           3538572    0.048889   

    score_interes  
62       1.508707  
64       1.271153  
10       1.211578  
44       1.185939  
54       1.051130  
47       1.010157  
77       1.007374  
61       0.916355  
32       0.887952  


C:\Users\ADRIAA~1\AppData\Local\Temp/ipykernel_14764/1801132241.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\ADRIAA~1\AppData\Local\Temp/ipykernel_14764/1801132241.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



# Potential signings

In [ ]:
df.columns

Index(['jugador', 'diferencia_valor', 'valor_actual', 'valor_anterior',
       'puja_maxima_rentable', 'propietario', 'fecha'],
      dtype='object')

In [ ]:
import pandas as pd
import plotly.express as px

# ===============================
# 📊 Crear df_recommendations con assign
# ===============================
df_recommendations = (
    df_free
    .assign(
        # Valor proyectado en 14 días
        valor_14d = lambda x: x.valor_actual + (x.diferencia_valor * 14),

        # Ganancia por crecimiento esperado en 14 días
        ganancia_crecimiento = lambda x: x.valor_14d - x.valor_actual,

        # Ganancia máxima adicional por venta a la máquina (+10% sobre valor proyectado)
        ganancia_maquina = lambda x: x.valor_14d * 0.10,

        # Ganancia total esperada
        ganancia_total = lambda x: x.ganancia_crecimiento + x.ganancia_maquina
    )
)

# Normalizar score 0-1 según ganancia total
min_g = df_recommendations["ganancia_total"].min()
max_g = df_recommendations["ganancia_total"].max()
df_recommendations = df_recommendations.assign(
    score = lambda x: (x.ganancia_total - min_g) / (max_g - min_g),

    # Puja máxima y puja ideal basada en crecimiento proyectado
    puja_max = lambda x: x.valor_14d * 0.9,
    puja_ideal = lambda x: x.valor_actual + (x.valor_14d - x.valor_actual) * 0.6
)

# Lógica "no fichar": si puja_ideal < valor_actual
df_recommendations = df_recommendations.assign(
    puja_ideal_final = lambda x: x.puja_ideal.where(x.puja_ideal >= x.valor_actual, "no fichar")
)

# Seleccionar columnas finales
df_recommendations = df_recommendations[
    ["jugador", "valor_actual", "valor_14d", "puja_ideal_final", "puja_max", "score", "ganancia_total","ganancia_crecimiento","ganancia_maquina"]
]
df_recommendations

,jugador,valor_actual,valor_14d,puja_ideal_final,puja_max,score,ganancia_total,ganancia_crecimiento,ganancia_maquina
0,Antony,72380113,121920121,102104117.8,109728108.9,1.000000,61732020.1,49540008,12192012.1
3,Vinicius Junior,111716407,136342099,126491822.2,122707889.1,0.708269,38259901.9,24625692,13634209.9
4,Arda Guler,87166587,110964501,101445335.4,99868050.9,0.666440,34894364.1,23797914,11096450.1
9,Lamine Yamal,145535020,162403718,155656238.8,146163346.2,0.644251,33109069.8,16868698,16240371.8
10,Tajon Buchanan,14338053,29096951,23193391.8,26187255.9,0.452344,17668593.1,14758898,2909695.1
...,...,...,...,...,...,...,...,...,...
595,Isco Alarcon,40741893,23566259,no fichar,21209633.1,0.048561,-14819008.1,-17175634,2356625.9
596,Alejandro Balde,52679857,35090957,no fichar,31581861.3,0.057749,-14079804.3,-17588900,3509095.7
597,Pau Cubarsi,63677861,42773369,no fichar,38496032.1,0.026088,-16627155.1,-20904492,4277336.9
598,Alex Baena,53946766,32018748,no fichar,28816873.2,0.000000,-18726143.2,-21928018,3201874.8


In [ ]:
df_free

,jugador,diferencia_valor,valor_actual,valor_anterior,puja_maxima_rentable,propietario,fecha,diferencia_valor_fmt,valor_actual_fmt,valor_anterior_fmt,puja_maxima_rentable_fmt,eficiencia,score_interes
0,Antony,3538572,72380113,68841541,0,,2025-09-04,3.538.572,72.380.113,68.841.541,0,0.048889,0.884761
3,Vinicius Junior,1758978,111716407,109957429,102494422,,2025-09-04,1.758.978,111.716.407,109.957.429,102.494.422,0.015745,0.291779
4,Arda Guler,1699851,87166587,85466736,79752866,,2025-09-04,1.699.851,87.166.587,85.466.736,79.752.866,0.019501,0.356546
9,Lamine Yamal,1204907,145535020,144330113,142275318,,2025-09-04,1.204.907,145.535.020,144.330.113,142.275.318,0.008279,0.155614
10,Tajon Buchanan,1054207,14338053,13283846,7339758,,2025-09-04,1.054.207,14.338.053,13.283.846,7.339.758,0.073525,1.211578
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,Isco Alarcon,-1226831,40741893,41968724,50731491,,2025-09-04,-1.226.831,40.741.893,41.968.724,50.731.491,-0.030112,-0.527650
596,Alejandro Balde,-1256350,52679857,53936207,60355527,,2025-09-04,-1.256.350,52.679.857,53.936.207,60.355.527,-0.023849,-0.424025
597,Pau Cubarsi,-1493178,63677861,65171039,72442549,,2025-09-04,-1.493.178,63.677.861,65.171.039,72.442.549,-0.023449,-0.421362
598,Alex Baena,-1566287,53946766,55513053,65711090,,2025-09-04,-1.566.287,53.946.766,55.513.053,65.711.090,-0.029034,-0.516906


In [ ]:
fig = px.bar(
    df_recommendations.sort_values("score", ascending=False),
    x="jugador",
    y="score",
    color="score",
    hover_data=["valor_actual", "valor_14d", "puja_ideal_final", "puja_max","ganancia_total"],
    title="Jugadores libres: ranking por score (rentabilidad relativa)",
    labels={
        "jugador": "Jugador",
        "score": "Score (0-1)"
    }
)

fig.show()

# Daily graphs

In [ ]:
import os
import pandas as pd

#Pilla folder path
data_path = 'Data'

# empty df to fill in
daily_dfs = []

# Get all the df in the folder that end up with .csv
for filename in os.listdir(data_path):
    # make sure it's a csv
    if filename.endswith('.csv'):
        file_path = os.path.join(data_path, filename)
        
        # add it in the final df.
        df = pd.read_csv(file_path)
        daily_dfs.append(df)

# Put together all the csvs
if daily_dfs:
    df_daily = pd.concat(daily_dfs, ignore_index=True)

       # 🔧 FIX: asegurar que diferencia_valor es numérico
    df_daily["diferencia_valor"] = (
        df_daily["diferencia_valor"]
        .astype(str)                               # todo a string
        .str.replace(r"\.", "", regex=True)        # quitar separadores de miles
        .str.replace(",", ".", regex=False)        # cambiar coma decimal por punto (si existe)
        .astype(float)                             # convertir a float
    )
    print("¡Listo! El DataFrame 'df_daily' ha sido creado.")
    print(f"El DataFrame tiene {df_daily.shape[0]} filas y {df_daily.shape[1]} columnas.")
else:
    print("No se encontraron archivos CSV en la carpeta 'Data'.")

¡Listo! El DataFrame 'df_daily' ha sido creado.
El DataFrame tiene 1793 filas y 11 columnas.


In [ ]:

import plotly.express as px

# df aggregated
df_daily_graph = (
    df_daily
    .query('propietario != ""')
    .groupby(['fecha', 'propietario'])['diferencia_valor']
    .sum()
    .reset_index()
)

# graph code
fig = px.bar(
    df_daily_graph,
    x='fecha',
    y='diferencia_valor',
    barmode='group',
    color='propietario',
    title='Diferencia de Valor de Jugadores por Propietario a lo Largo del Tiempo',
    labels={
        "fecha": "Fecha",
        "diferencia_valor": "Diferencia de Valor",
        "propietario": "Propietario"
    }
)

fig.show()

In [ ]:
(df_daily
    .query('propietario != ""'
           '& fecha == "2025-09-04"')
    .groupby(['fecha', 'propietario'])['diferencia_valor']
    .sum()
    .reset_index()
)

,fecha,propietario,diferencia_valor
0,2025-09-04,Adri,1.932.6421.580.315661.236601.279430.388389.138...
1,2025-09-04,Carre,1.678.7091.478.6981.465.9931.048.869895.970830...
2,2025-09-04,Enric,825.066630.163572.867377.481261.697220.222113....
3,2025-09-04,Swedish,1.889.915713.723635.161540.127454.480432.28242...


In [ ]:
df_daily.dtypes

jugador                     object
diferencia_valor            object
valor_actual                object
valor_anterior              object
puja_maxima_rentable        object
propietario                 object
fecha                       object
diferencia_valor_fmt        object
valor_actual_fmt            object
valor_anterior_fmt          object
puja_maxima_rentable_fmt    object
dtype: object